# 9.7 AdaBoost (Adaptive Boosting)

**Machine Learning Mastery: Mathematics to MLOps** · Chapter 9: Ensemble Learning

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RithikRaj129/ml-mastery/blob/main/chapter-09-ensemble-learning/09.07_adaboost.ipynb)

Companion code for Section 9.7 of the book. Run the cells in order from top to bottom.

**Example 9.7.1**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.datasets import load_breast_cancer, make_classification
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score
from sklearn.base import clone
import warnings; warnings.filterwarnings('ignore')

# ── AdaBoost from scratch ─────────────────────────────────────────
class AdaBoostScratch:
    '''AdaBoost classifier from scratch (SAMME algorithm).'''

    def __init__(self, n_estimators=50, learning_rate=1.0):
        self.n_estimators = n_estimators
        self.learning_rate= learning_rate
        self.stumps_      = []
        self.alphas_      = []
        self.errors_      = []
        self.weights_history_ = []

    def fit(self, X, y):
        n = len(X)
        # Convert to {-1, +1}
        y_pm = np.where(y == 0, -1, 1)
        w = np.ones(n) / n

        for m in range(self.n_estimators):
            # Train depth-1 stump with sample weights
            stump = DecisionTreeClassifier(max_depth=1)
            stump.fit(X, y_pm, sample_weight=w)
            pred  = stump.predict(X)

            # Weighted error
            err = np.sum(w[pred != y_pm]) / np.sum(w)
            err = np.clip(err, 1e-10, 1-1e-10)

            # Learner weight
            alpha = self.learning_rate * 0.5 * np.log((1-err)/err)

            # Update weights
            w = w * np.exp(-alpha * y_pm * pred)
            w = w / w.sum()   # Normalise

            self.stumps_.append(stump)
            self.alphas_.append(alpha)
            self.errors_.append(err)
            self.weights_history_.append(w.copy())

        return self

    def predict(self, X):
        pred = sum(alpha * stump.predict(X)
                   for alpha, stump in zip(self.alphas_, self.stumps_))
        return (pred >= 0).astype(int)

    def score(self, X, y):
        return (self.predict(X) == y).mean()

    def staged_accuracy(self, X, y):
        running_pred = np.zeros(len(X))
        accuracies   = []
        for alpha, stump in zip(self.alphas_, self.stumps_):
            running_pred += alpha * stump.predict(X)
            acc = ((running_pred >= 0).astype(int) == y).mean()
            accuracies.append(acc)
        return accuracies


# ── Train and evaluate ────────────────────────────────────────────
data = load_breast_cancer()
X, y = data.data, data.target
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# From scratch
ada_scratch = AdaBoostScratch(n_estimators=100, learning_rate=1.0)
ada_scratch.fit(X_tr, y_tr)
print(f'AdaBoost from scratch: {ada_scratch.score(X_te, y_te):.4f}')

# Sklearn comparison
print('\n=== AdaBoost Hyperparameter Analysis ===')
print(f'{'Config':<30} {'Train':>8} {'Test':>8} {'CV(5)':>8}')
print('='*58)

configs = [
    ('n=50, lr=1.0, depth=1',  AdaBoostClassifier(n_estimators=50,  learning_rate=1.0, random_state=42)),
    ('n=100,lr=1.0, depth=1',  AdaBoostClassifier(n_estimators=100, learning_rate=1.0, random_state=42)),
    ('n=200,lr=0.5, depth=1',  AdaBoostClassifier(n_estimators=200, learning_rate=0.5, random_state=42)),
    ('n=100,lr=1.0, depth=2',  AdaBoostClassifier(
        estimator=DecisionTreeClassifier(max_depth=2),
        n_estimators=100, learning_rate=1.0, random_state=42)),
    ('n=100,lr=1.0, depth=3',  AdaBoostClassifier(
        estimator=DecisionTreeClassifier(max_depth=3),
        n_estimators=100, learning_rate=1.0, random_state=42)),
]

for name, model in configs:
    model.fit(X_tr, y_tr)
    tr = model.score(X_tr, y_tr)
    te = model.score(X_te, y_te)
    cv = cross_val_score(clone(model), X, y, cv=5, scoring='accuracy').mean()
    print(f'{name:<30} {tr:>8.4f} {te:>8.4f} {cv:>8.4f}')

# ── Visualise stump weights (α) and errors over rounds ────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('AdaBoost Analysis', fontsize=14, fontweight='bold')

rounds = range(1, len(ada_scratch.alphas_)+1)

ax = axes[0]
ax.plot(rounds, ada_scratch.alphas_, 'b-o', ms=4, lw=2)
ax.set_xlabel('Boosting Round'); ax.set_ylabel('Alpha (Learner Weight)')
ax.set_title('Stump Weight (α) per Round\n(high α = low error stump)')
ax.grid(alpha=0.3)

ax = axes[1]
ax.plot(rounds, ada_scratch.errors_, 'r-o', ms=4, lw=2)
ax.axhline(0.5, color='gray', ls='--', alpha=0.6, label='Random (ε=0.5)')
ax.set_xlabel('Boosting Round'); ax.set_ylabel('Weighted Error (ε)')
ax.set_title('Stump Error per Round\n(stumps focus on hard examples)')
ax.legend(); ax.grid(alpha=0.3)

ax = axes[2]
train_staged = ada_scratch.staged_accuracy(X_tr, y_tr)
test_staged  = ada_scratch.staged_accuracy(X_te, y_te)
ax.plot(rounds, train_staged, 'b-', lw=2, alpha=0.8, label='Train')
ax.plot(rounds, test_staged,  'r-', lw=2, alpha=0.8, label='Test')
best_r = np.argmax(test_staged)+1
ax.axvline(best_r, color='green', ls='--', lw=2, label=f'Best round={best_r}')
ax.set_xlabel('Boosting Round'); ax.set_ylabel('Accuracy')
ax.set_title('Staged Accuracy: Train vs Test')
ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('adaboost_analysis.png', dpi=150, bbox_inches='tight')
plt.show()